# Invariance Experiment — Target–Background Spectral Correlation

**Claim:** our spatial score detector (CF-Attn) is *invariant* to how spectrally correlated the target is with the background. Pure spectral detectors (THANTD, AMF) only win when the target is spectrally **separable**; they collapse to near-random when the target is **entangled** (a subpixel additive target whose signature is a linear combination of the local background materials).

We use the **same heterogeneous manual boxes** as the main experiment (a spatially mixed background is the whole challenge), hold the box and the additive model fixed, and vary ONLY the target's spectral **direction** (all signatures renormalized to the same L2 norm → matched amplitude). PCA is fit on the whole image (fine for this controlled experiment — it only defines the shared feature space).

| regime | signature | residual outside scene-material span |
|---|---|---|
| **A: entangled** | 0.8·(dominant class) + 0.2·patch_mean | ≈ 0 (the original target — a scene material) |
| **B: distinct** | mean of a real class absent from the box | mild in Pavia (materials are low-rank) |
| **C: synth-perp** | synthetic, orthogonal to all scene materials | ~1.0 (genuinely out of scene) |

**Expected:** CF-Attn high in **all** regimes (invariant); THANTD/AMF low in A, recovering toward C as the target becomes separable.

In [ ]:
# ── Cell 1: Clone/pull repo + deps ────────────────────────────────────────
import os, sys
REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'
if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} pull
else:
    !git clone {REPO_URL} {LOCAL_PROJECT}
!pip install -q scikit-learn scipy tqdm matplotlib pyyaml
for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())

In [ ]:
# ── Cell 2: Mount Drive (data + results) ────────────────────────────────
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/proj/data', exist_ok=True)
DATA_DST = '/content/proj/data/pavia-u.mat'
if not os.path.exists(DATA_DST):
    for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
        if os.path.exists(src):
            os.symlink(src, DATA_DST); print('Data linked:', src); break
    else:
        raise FileNotFoundError('Upload pavia-u.mat to /MyDrive/final_paper/')
else:
    print('Data already present.')
RESULTS_DIR = '/content/drive/MyDrive/final_paper/invariance_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results dir:', RESULTS_DIR)

In [ ]:
# ── Cell 3a: DRY RUN (quick, no THANTD) ───────────────────────────────
RESULTS_DIR = '/content/drive/MyDrive/final_paper/invariance_results'
!python -u experiments/spatial/run_invariance.py \
    --config experiments/spatial/colab.yaml \
    --results_dir {RESULTS_DIR} \
    --dry-run --no-thantd

In [ ]:
# ── Cell 3b: FULL RUN (4 heterogeneous manual boxes × 3 regimes, all detectors) ───
# Uses the SAME challenging manual_boxes.json scenarios as the main experiment.
# 3 target regimes per box: A entangled, B distinct real class, C synth-orthogonal.
RESULTS_DIR = '/content/drive/MyDrive/final_paper/invariance_results'
!python -u experiments/spatial/run_invariance.py \
    --config experiments/spatial/colab.yaml \
    --results_dir {RESULTS_DIR}

In [ ]:
# ── Cell 4: Display figures + summary table ────────────────────────────
import os, glob, json
import matplotlib.pyplot as plt, matplotlib.image as mpimg
RESULTS_DIR = '/content/drive/MyDrive/final_paper/invariance_results'
runs = sorted(glob.glob(os.path.join(RESULTS_DIR, 'invariance_*')))
assert runs, 'No results found. Run Cell 3b first.'
run_dir = runs[-1]
print('Showing:', run_dir)
ap = os.path.join(run_dir, 'all_invariance_auc.json')
if os.path.exists(ap):
    auc = json.load(open(ap))
    for sk, a in auc.items():
        print(f'\n=== {sk} ===')
        regimes = list(a.keys()); dets = list(next(iter(a.values())).keys())
        print('  ' + 'detector'.ljust(14) + ''.join(r.rjust(16) for r in regimes))
        for d in dets:
            print('  ' + d.ljust(14) + ''.join(f'{a[r][d]:16.3f}' for r in regimes))
!apt-get install -y -q poppler-utils
for pdf in sorted(glob.glob(os.path.join(run_dir, 'scenario_*', 'figures', '*.pdf'))):
    png = pdf.replace('.pdf', '.png')
    os.system(f'pdftoppm -r 150 -png -singlefile "{pdf}" "{pdf[:-4]}"')
    if os.path.exists(png):
        img = mpimg.imread(png)
        fig, ax = plt.subplots(figsize=(13, 6))
        ax.imshow(img); ax.axis('off')
        ax.set_title('/'.join(pdf.split('/')[-3:]), fontsize=9)
        plt.tight_layout(); plt.show()